# Light: references for heavy models on Colab

Light (https://github.com/sushvibez/light) verifies every model port against the transformers implementation on real weights. The development machine has a 16 GB card, so checkpoints that only fit there as a quantized GGUF (8B–30B) get their reference numbers from this notebook instead: it runs `scripts/verify/ref_json.py` on Colab's GPU and produces one JSON per model that `models/heavy_ref_test.go` compares the local GGUF against.

Runtime: any GPU works for the references (T4 16 GB holds 8B in 8-bit, A100 40 GB holds 12B–30B in BF16). To run Light's own test suites on Colab use `light_heavy_tests.ipynb` next to this file.

In [ ]:
#@title 1. Setup
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install -U transformers accelerate bitsandbytes sentencepiece huggingface_hub
!git clone -q https://github.com/sushvibez/light.git /content/light
import os
from huggingface_hub import login
# Gated repos (Llama, Gemma, Mistral): paste a token with read access.
if os.environ.get('HF_TOKEN'):
    login(os.environ['HF_TOKEN'])
else:
    login()
os.makedirs('/content/refs', exist_ok=True)

In [ ]:
#@title 2. Produce references
# (repo, dtype, GGUF file glob the local test should verify, tolerance override)
# dtype: bf16 needs ~2 bytes/param of VRAM; 8bit/4bit (bitsandbytes) fit bigger models on a T4
# but perturb the logits, so give them a wider tolerance.
MODELS = [
    ("meta-llama/Llama-3.1-8B-Instruct", "bf16", "llama-3.1-8b-instruct-q4_k_m.gguf", 0),
    ("mistralai/Mistral-7B-Instruct-v0.3", "bf16", "mistral-7b-instruct-v0.3-q4_k_m.gguf", 0),
    ("Qwen/Qwen3-8B", "bf16", "qwen3-8b-q4_k_m.gguf", 0),
    ("google/gemma-3-12b-it", "bf16", "gemma-3-12b-it-q4_k_m.gguf", 0),
    # ("Qwen/Qwen3-30B-A3B", "8bit", "qwen3-30b-a3b-q3_k_m.gguf", 3.0),   # A100 only
]
import subprocess, torch
free_gb = torch.cuda.mem_get_info()[0] / 2**30
print(f'{free_gb:.0f} GB free on the GPU')
for repo, dtype, gguf, tol in MODELS:
    out = f"/content/refs/{repo.split('/')[-1].lower()}.json"
    cmd = ['python', '/content/light/scripts/verify/ref_json.py', repo, '--dtype', dtype, '--gguf', gguf, '--out', out]
    if tol:
        cmd += ['--tolerance', str(tol)]
    print('\n==>', ' '.join(cmd))
    subprocess.run(cmd, check=False)
    torch.cuda.empty_cache()

In [ ]:
#@title 2b. Download the references
# Copy the files into models/testdata/refs/ of your checkout, put the matching
# GGUF files (same names as the globs above) in ~/.cache/light/gguf and run:
#   LIGHT_HEAVY_TESTS=1 go test ./models -run TestGGUFAgainstReferenceJSON -v
!cd /content/refs && zip -q -r /content/light-refs.zip .
from google.colab import files
files.download('/content/light-refs.zip')